# ocean: line

ocean transport data cross defined lines

**coordinate**: tavg-u-ht-sea
- tavg: time average
- u: ocean surface
- ht: defined lines
- sea: ocean domain

In [ ]:
## Import libraries
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import sys
import os
import glob

sys.path.append(os.getcwd())
from utils import read_variables, read_compound_name

In [ ]:
# parameters for the cmorized data
cmorout=''                  # root for cmorized data, e.g., '/scratch/$USER/cmorout'
source_id      = ''         # model name, e.g., 'NorESM3-LM'
experiment_id  = ''         # experiment name, e.g., 'historical', 'ssp585', 'piControl'
variant_label  = ''         # variant label, e.g., 'r1i1p1f1'
grid_label     = ''         # grid label, e.g., 'gn', 'gr', 'g999'
version        = ''         # version, e.g., 'v20260601'

In [ ]:
# data path
data_path = os.path.join(cmorout, source_id, experiment_id, version)

# load methods for plotting and set defaults
methods =read_variables('data/methods.txt')
#print(methods.keys())

**List of data validated:**

In [ ]:
# load compound names

coords = ('tavg-u-ht-sea')
cnames = read_compound_name()
# examples of compound names:
# cnames = ['ocean.hfacrossline.tavg-u-ht-sea.mon.glb', 'ocean.mfo.tavg-u-ht-sea.mon.glb', 'ocean.sfacrossline.tavg-u-ht-sea.mon.glb']
for cname in cnames:
    realm = cname.split('.')[0]
    coord = cname.split('.')[2]
    if realm != 'ocean' or coord not in coords:
        continue
    else:
        print(cname)


In [ ]:
# loop through compound names and plot
for cname in cnames:
    mth_vert = 'mean'
    mth_ts = 'mean'
    mth_cmap = 'mpl.colormaps["viridis"]'
    if cname not in methods.keys():
        print(f"{cname} not found in methods.txt, using default methods for plotting.")
    else:
        if methods[cname] is not None:
            if 'vertical' in methods[cname].keys():
                mth_vert = methods[cname]['vertical']

            if 'timeseries' in methods[cname].keys():
                mth_ts = methods[cname]['timeseries']

            if 'cmap' in methods[cname].keys():
                mth_cmap = methods[cname]['cmap']

    realm = cname.split('.')[0]
    var = cname.split('.')[1]
    coord = cname.split('.')[2]
    freq = cname.split('.')[3]

    if realm != 'ocean' or coord not in coords:
        continue

    data_file = var+'_'+coord+'_*_'+grid_label+'_'+source_id+'_'+experiment_id+'_'+variant_label+'_*.nc'

    if not glob.glob(os.path.join(data_path, data_file)):
        continue

    #with xr.open_mfdataset(os.path.join(data_path, data_file)) as ds:
    data_file = glob.glob(os.path.join(data_path, data_file))[0]
    with xr.open_dataset(os.path.join(data_path, data_file)) as ds:
        if var in ds:
            data = ds[var]
        else:
            continue

    print(f'\033[1m{cname}\033[0m')
    print(f'long name: {data.long_name} ({data.units})')
    
    nlines = data.line.size
    sector = ds['sector'].data
    fig, axs = plt.subplots((nlines+1)//2, 2, figsize=(12, 16), dpi=96, sharex=True, sharey=False)
    ax = axs.flatten()
    for i in range(nlines):
        data.isel(line=i).plot(ax=ax[i])
        data.isel(line=i).plot(ax=ax[i])
        ax[i].set_title(f'{sector[i].astype(str).strip()}')
        ax[i].set_ylabel(data.units)
        ax[i].set_xlabel('')
        ax[i].grid()

    # Turn off x‑axis ticks for all subplots
    for ax in axs.flatten():
        ax.xaxis.set_tick_params(labelbottom=False, bottom=False)

    # Turn on x‑axis ticks only for the bottom row (last two axes)
    for ax in axs[-1,:]:
        ax.xaxis.set_tick_params(labelbottom=True, bottom=True)
        ax.set_xlabel('Time')

    fig.suptitle(f"{data.long_name}", fontsize=16)

    plt.tight_layout()
    plt.show()

    del data
    del ax, axs, fig